https://massive.com/pricing?product=options

To gain access to hourly data from the same API, would require $199 / month.

I am thinking about solutions, such as using a different API (but limitations and unreliability led me to Massive in the first place)

In [1]:
import requests
import pandas as pd
from pathlib import Path
from datetime import date, datetime, timedelta
from dateutil.relativedelta import relativedelta

# Bypass scientific notation
pd.options.display.float_format = '{:,.0f}'.format


def load_api_key(filepath="api_keys/massive.txt"):
    """Load Massive API key from a local text file."""
    return Path(filepath).read_text(encoding="utf-8").strip()


def get_all_pages(url, params=None):
    """Fetch all paginated results from Massive."""
    all_results = []

    while url:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        if data.get("status") == "NOT_AUTHORIZED":
            raise PermissionError(data.get("message", "Not authorized for this request."))

        results = data.get("results", [])
        all_results.extend(results)

        # next_url can be used to fetch the next page
        url = data.get("next_url")
        params = None

    return all_results


def get_massive_hourly_bars(
    ticker="AAPL",
    api_key_path="api_keys/massive.txt",
    years=1,
    safety_days=1,
    adjusted=True,
    sort="asc"
):
    """
    Fetch hourly aggregate bars for a ticker from Massive.

    Parameters
    ----------
    ticker : str
        Stock ticker symbol, e.g. 'AAPL'
    api_key_path : str
        Path to text file containing API key
    years : int
        Number of years of history to request
    safety_days : int
        Number of days to move forward from exact cutoff to avoid entitlement edge issues
    adjusted : bool
        Whether to return adjusted prices
    sort : str
        'asc' or 'desc'

    Returns
    -------
    pd.DataFrame
    """
    api_key = load_api_key(api_key_path)

    today = date.today()
    start_date = today - relativedelta(years=years) + timedelta(days=safety_days)

    start_str = start_date.isoformat()
    end_str = today.isoformat()

    url = f"https://api.massive.com/v2/aggs/ticker/{ticker}/range/1/hour/{start_str}/{end_str}"

    params = {
        "adjusted": str(adjusted).lower(),
        "sort": sort,
        "limit": 50000,
        "apiKey": api_key,
    }

    results = get_all_pages(url, params)

    if not results:
        raise ValueError(f"No hourly data returned for {ticker}.")

    df = pd.DataFrame(results).rename(columns={
        "t": "timestamp",
        "o": "open",
        "h": "high",
        "l": "low",
        "c": "close",
        "v": "volume",
        "vw": "vwap",
        "n": "transactions",
    })

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df["ticker"] = ticker

    preferred_order = [
        "timestamp", "ticker", "open", "high", "low", "close",
        "volume", "vwap", "transactions"
    ]
    df = df[[col for col in preferred_order if col in df.columns]]

    return df

In [2]:
aapl_hourly = get_massive_hourly_bars(
    ticker="AAPL",
    api_key_path="api_keys/massive.txt",
    years=1,
    safety_days=1,
    adjusted=True,
    sort="asc"
)

aapl_hourly.to_csv("datasets/aapl_hourly.csv", index=False)

print(aapl_hourly.head())
print(aapl_hourly.tail())
print(aapl_hourly.shape)
print("Date range:", aapl_hourly["timestamp"].min(), "to", aapl_hourly["timestamp"].max())

HTTPError: 401 Client Error: Unauthorized for url: https://api.massive.com/v2/aggs/ticker/AAPL/range/1/hour/1753286400000/2026-04-21?cursor=bGltaXQ9NTAwMDAmc29ydD1hc2M

In [3]:
print(results[:3])

NameError: name 'results' is not defined